# Actividad 3 · Ensamblando la molécula
### Modelo didáctico de términos de campo de fuerza
**Taller:** Física y matemática detrás de los campos de fuerza y la dinámica molecular · XXXIV Semana de la Química, Bogotá 2026

---

**Duración:** ~25 min · **Herramienta:** Google Colab o Jupyter · **Prerrequisito:** Actividades 1 y 2

**Objetivos:**
1. Reconocer cómo un modelo didáctico puede ilustrar la suma de términos de energía.
2. Construir un modelo simplificado para una molécula triatómica con enlaces, ángulo y Lennard-Jones.
3. Descubrir que los términos **compiten** y que el mínimo real es un compromiso.
4. Distinguir este modelo simplificado de un campo de fuerza parametrizado como MMFF94.

## Concepto clave

Como esquema general, un campo de fuerza puede incluir términos como los siguientes. La expresión es ilustrativa y no representa la forma funcional ni el conjunto completo de términos de MMFF94:

$$U = \underbrace{\sum k_b(r-r_0)^2}_{\text{enlaces}} + \underbrace{\sum k_\theta(\theta-\theta_0)^2}_{\text{\'angulos}} + \underbrace{\sum V_n[1+\cos(n\phi-\gamma)]}_{\text{diedros}} + \underbrace{\sum 4\varepsilon\left[\left(\tfrac{\sigma}{r}\right)^{12}-\left(\tfrac{\sigma}{r}\right)^{6}\right]}_{\text{vdW}} + \underbrace{\sum \frac{q_i q_j}{4\pi\varepsilon_0 r_{ij}}}_{\text{electrost\'atica}}$$

- Los términos de enlace y ángulo suelen incluir contribuciones internas parametrizadas.
- LJ es una opción común para representar ciertas interacciones de van der Waals; no es la única forma funcional posible.
- MMFF94 define un conjunto específico de funciones, reglas y parámetros ajustados para sistemas moleculares. No se reduce a cambiar los parámetros de una fórmula universal.

> **Alcance de esta actividad:** las expresiones y parámetros usados aquí son un modelo didáctico simplificado, inspirado en términos habituales de campos de fuerza; no reproducen MMFF94.

## Paso 0 · Preparativos

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Paso 1 · El enlace químico es un resorte: solución

En los campos de fuerza, un enlace químico se modela como un **resorte**:

$$U_{\text{enlace}}(r) = k\,(r - r_0)^2$$

- $r_0$: longitud de equilibrio del enlace (donde $U = 0$, el resorte "descansado").
- $k$: rigidez del enlace. Enlaces reales tienen $k$ grande ($\sim$ 300–700 kcal/mol/Å²): **difíciles de estirar o comprimir**.

> Derivación: ¿cuál es la fuerza del enlace, $F = -dU/dr$? La expresión esperada es $F = -2k(r-r_0)$, proporcional al estiramiento y con signo restaurador.

Implementación del potencial: en Python, elevar al cuadrado se expresa como `x**2`.

In [ ]:
def U_enlace(r, k, r0):
    """Energía de un enlace: resorte armónico U = k*(r - r0)^2."""
    # Solución: return ...
    return k * (r - r0)**2

# Prueba: un enlace con k = 500 kcal/mol/A^2 y r0 = 1.1 A
print("U(1.1) =", U_enlace(1.1, 500.0, 1.1), " <- en el equilibrio (debe ser 0)")
print("U(1.2) =", U_enlace(1.2, 500.0, 1.1), " <- estirado 0.1 A")
print("U(1.0) =", U_enlace(1.0, 500.0, 1.1), " <- comprimido 0.1 A (misma energía: simetría del resorte)")

## Paso 2 · Los otros términos y la molécula completa: solución

Se considera una molécula triatómica **A–B–C**, cuya energía en este modelo simplificado suma cuatro contribuciones:

1. Enlace A–B (resorte)
2. Enlace B–C (resorte)
3. Ángulo A–B–C (resorte angular: $k_\alpha(\theta-\theta_0)^2$)
4. Un término LJ entre **A y C** para ilustrar la competencia entre contribuciones energéticas. A y C forman un par 1–3, separados por el átomo B; el término se incluye como simplificación didáctica. En distintos campos de fuerza, las interacciones 1–3 pueden excluirse o tratarse con reglas especiales.

La geometría: B en el origen, A sobre el eje $x$ a distancia $r_1$, C a distancia $r_2$ formando ángulo $\theta$ en B. Por trigonometría:
$$AC = \sqrt{r_1^2 + r_2^2 - 2\,r_1 r_2 \cos\theta}$$

Las funciones de ángulo, Lennard-Jones y distancia ya están implementadas. Completar la suma de términos en `U_molecula`.

In [ ]:
def U_angulo(theta, ka, th0):
    """Resorte angular: ka*(theta - th0)^2 con ángulos en grados."""
    return ka * (np.radians(theta) - np.radians(th0))**2

def U_LJ(r, eps, sig):
    """Lennard-Jones (Actividad 1)."""
    return 4 * eps * ((sig / r)**12 - (sig / r)**6)

def distancia_AC(r1, r2, theta):
    """Distancia A...C por ley de cosenos (theta en grados)."""
    th = np.radians(theta)
    return np.sqrt(r1**2 + r2**2 - 2*r1*r2*np.cos(th))

# ---- Parámetros de nuestra "molécula" (valores estilo real, kcal/mol, A, grados) ----
k_en, r0_en = 500.0, 1.1     # enlaces A-B y B-C: rígidos, 1.1 A
k_ang, th0  = 80.0, 120.0    # ángulo A-B-C
eps, sig    = 0.3, 2.2       # Lennard-Jones entre A y C

def U_molecula(r1, r2, theta):
    """Energía total de la molécula A-B-C: SUMA de los cuatro términos."""
    # Solución: return U_enlace(...) + U_enlace(...) + U_angulo(...) + U_LJ(...)
    return (U_enlace(r1, k_en, r0_en) + U_enlace(r2, k_en, r0_en)
            + U_angulo(theta, k_ang, th0) + U_LJ(distancia_AC(r1, r2, theta), eps, sig))

print("U(1.1, 1.1, 120) =", round(U_molecula(1.1, 1.1, 120.0), 4), "kcal/mol  <- geometría de referencia")

## Experimento 1 · ¿Dónde está el mínimo? Los términos compiten

Exploración del paisaje molecular: fijar $r_1 = r_2 = 1.1$ Å y variar el ángulo; luego fijar $\theta = 120°$ y variar $r_1$.

**Predicción previa:** ¿se espera que el mínimo esté exactamente en $\theta = 120°$ y $r = 1.1$ Å (los valores de equilibrio de los resortes), o que se desplace? El término LJ entre A y C favorece una mayor separación, mientras que los resortes favorecen sus valores de equilibrio.

In [ ]:
# --- Paisaje angular ---
thetas = np.linspace(60, 180, 400)
U_th = U_molecula(r0_en, r0_en, thetas)
i_th = np.argmin(U_th)

# --- Paisaje del enlace ---
r1s = np.linspace(0.8, 1.8, 400)
U_r1 = U_molecula(r1s, r0_en, th0)
i_r = np.argmin(U_r1)

print(f"Mínimo angular:  theta = {thetas[i_th]:.1f}°   (resorte solo pediría {th0}°)")
print(f"Mínimo de enlace: r1  = {r1s[i_r]:.3f} Å  (resorte solo pediría {r0_en} Å)")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
axes[0].plot(thetas, U_th, color='darkred', lw=2)
axes[0].axvline(th0, color='gray', ls='--', lw=1, label=f'resorte solo: {th0}°')
axes[0].axvline(thetas[i_th], color='green', ls=':', lw=1.5, label=f'mínimo real: {thetas[i_th]:.1f}°')
axes[0].set_xlabel('ángulo θ [°]'); axes[0].set_ylabel('U [kcal/mol]')
axes[0].set_title('Los términos compiten: el ángulo real no es 120°')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(r1s, U_r1, color='navy', lw=2)
axes[1].axvline(r0_en, color='gray', ls='--', lw=1, label=f'resorte solo: {r0_en} Å')
axes[1].axvline(r1s[i_r], color='green', ls=':', lw=1.5, label=f'mínimo real: {r1s[i_r]:.3f} Å')
axes[1].set_xlabel('distancia r1 [Å]'); axes[1].set_ylabel('U [kcal/mol]')
axes[1].set_title('Idem para la longitud de enlace')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

print("\n Lectura: el mínimo real difiere de los valores de los resortes porque el término LJ A...C favorece una mayor separación.")

## Experimento 2 · Relajación de una geometría: analogía didáctica

Una geometría inicial puede no estar en un mínimo de energía. En un flujo molecular real, un programa puede minimizarla con un campo de fuerza como MMFF94. El ejemplo siguiente ilustra la relajación, pero **no implementa MMFF94**: utiliza el modelo didáctico de esta actividad y optimiza solo la coordenada $r_1$.

Se reutiliza el descenso de gradiente de la Actividad 2; la derivada de esta energía simplificada se calcula numéricamente mediante diferencias finitas:

$$F(r_1) \approx -\frac{U(r_1 + h) - U(r_1 - h)}{2h}$$

Se parte de un enlace A–B estirado a 1.5 Å y se permite su relajación. *Completar el paso de actualización* como en la Actividad 2.

In [ ]:
def fuerza_numerica(f, x, h=1e-4):
    """Fuerza = -dU/dx por diferencias finitas (la computadora deriva por nosotros)."""
    return -(f(x + h) - f(x - h)) / (2 * h)

# Energía vista como función solo de r1 (con r2 y theta fijos en sus valores de referencia)
def U_solo_r1(r1):
    return U_molecula(r1, r0_en, th0)

r = 1.5            # enlace estirado (geométricamente "mal")
alpha = 0.0005     # paso pequeño: las energías moleculares cambian rápido
trayectoria = [r]
for paso in range(20000):
    F = fuerza_numerica(U_solo_r1, r)
    # Solución: actualiza r (descenso de gradiente, igual que en la Actividad 2)
    r = r + alpha * F
    trayectoria.append(r)
    if abs(F) < 1e-10:
        break

print(f"Relajación convergió en {len(trayectoria)-1} pasos")
print(f"r1 final = {r:.4f} Å   (coincide con el mínimo del Experimento 1: {r1s[i_r]:.3f} Å)")
print(f"Energía final = {U_solo_r1(r):.4f} kcal/mol")

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(r1s, U_r1, color='gray', lw=1.5, label='paisaje U(r1)')
ax.plot(trayectoria, U_solo_r1(np.array(trayectoria)), 'o-', color='darkred',
        ms=4, lw=1, label='relajación (descenso de gradiente)')
ax.annotate('geométria inicial\n(enlace estirado)', xy=(1.5, U_solo_r1(1.5)), xytext=(1.55, 12),
            arrowprops=dict(arrowstyle='->', color='black'))
ax.set_xlabel('r1 [Å]'); ax.set_ylabel('U [kcal/mol]')
ax.set_title('Relajación de una coordenada en el modelo didáctico')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Experimento 3 · Efecto de los parámetros en el modelo

El experimento modifica los parámetros del **modelo didáctico** en `U_molecula_B` (enlace más blando y corto: $k=200$, $r_0=0.95$ Å; ángulo de $109.5°$). La comparación ilustra el efecto de los parámetros, pero **no es una parametrización de otra molécula ni una implementación de MMFF94**. MMFF94 posee funciones, reglas y parámetros propios para distintos entornos químicos.

In [ ]:
# Misma función, otros parámetros = OTRA MOLÉCULA
def U_molecula_B(r1, r2, theta):
    # átomos más pequeños (sigma menor), acorde con su enlace más corto
    return (U_enlace(r1, 200.0, 0.95) + U_enlace(r2, 200.0, 0.95)
            + U_angulo(theta, 40.0, 109.5) + U_LJ(distancia_AC(r1, r2, theta), 0.3, 1.9))

r1s_B = np.linspace(0.7, 1.6, 400)
U_B = U_molecula_B(r1s_B, 0.95, 109.5)
i_B = np.argmin(U_B)
print(f"Conjunto de parámetros B: mínimo de enlace en r1 = {r1s_B[i_B]:.3f} Å (r0 = 0.95 Å; el término LJ desplaza el mínimo)")

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(r1s, U_r1, color='darkred', lw=2, label='Modelo A: k=500, r0=1.1 Å (enlace rígido, largo)')
ax.plot(r1s_B, U_B, color='navy', lw=2, label='Modelo B: k=200, r0=0.95 Å (enlace blando, corto)')
ax.set_xlabel('r1 [Å]'); ax.set_ylabel('U [kcal/mol]')
ax.set_title('Mismas funciones, distintos parámetros → distintos paisajes de energía')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

print("\nNota: este modelo didáctico usa funciones simplificadas y no reproduce MMFF94.")

## Síntesis · El campo de fuerza pieza por pieza

| Término | Función | Significado | Parámetros (ej. MMFF94) |
|---|---|---|---|
| Enlace | $k(r-r_0)^2$ | resorte entre átomos unidos | $k$, $r_0$ por tipo de enlace |
| Ángulo | $k_\alpha(\theta-\theta_0)^2$ | resorte angular | $k_\alpha$, $\theta_0$ |
| Diedro | $V_n[1+\cos(n\phi-\gamma)]$ | rotación alrededor de enlaces | $V_n$, $n$, $\gamma$ |
| van der Waals | Por ejemplo, Lennard-Jones | atracción y repulsión | según el campo de fuerza |
| Electroestática | $q_i q_j / r_{ij}$ | cargas parciales | cargas $q_i$ |

> **Alcance:** las funciones, reglas de exclusión y parámetros varían entre campos de fuerza. La tabla resume familias de términos y no especifica MMFF94.

## Cierre · Conexión con el *docking molecular*

1. **Minimización de geometría.** Un programa puede relajar un ligando con MMFF94; el Experimento 2 solo ilustra el concepto con un modelo simplificado de una coordenada y no reproduce ese campo de fuerza.

2. **Preparación del ligando.** La protonación, tautomería y geometría iniciales afectan los resultados de la minimización y del docking; la minimización local no corrige necesariamente una elección química inicial incorrecta.

3. **Selección del campo de fuerza.** MMFF94 se diseñó para moléculas orgánicas pequeñas; otros campos, como AMBER, CHARMM y OPLS, tienen parametrizaciones y dominios de aplicación propios.

4. **¿Dónde estamos en el mapa del taller?**
   - Actividad 1: del potencial a la fuerza.
   - Actividad 2: paisaje energético, minimización y optimización.
   - **Actividad 3:** ensamblaje molecular y campo de fuerza estilo MMFF94.
   - Actividades 4–5: mover los átomos en el tiempo → dinámica molecular y auditoría de hipótesis de unión.

> **Pregunta guía del curso:** la evaluación de una hipótesis de unión comienza antes del docking, durante la preparación molecular. Queda descrito el papel del campo de fuerza en la minimización de un ligando.